<a href="https://colab.research.google.com/github/angelonels/Entity-Aware-Progressive-Query-Rewriting-for-Lightweight-Agentic-Retrieval-Augmented-Generation/blob/main/research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
!pip -q install datasets sentence-transformers rank-bm25 transformers accelerate evaluate scikit-learn spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 116.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [19]:
import re
import json
import math
import random
import string
import numpy as np
import pandas as pd
from dataclasses import dataclass
from collections import defaultdict, Counter
from typing import List, Dict, Tuple, Any

import torch
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics.pairwise import cosine_similarity
import evaluate
import spacy

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = 0 if torch.cuda.is_available() else -1
print("Using device:", "cuda" if device == 0 else "cpu")


@dataclass
class ExperimentConfig:
    train_examples_for_corpus: int = 1200
    validation_examples_for_corpus: int = 300
    evaluation_examples: int = 120

    retrieval_top_k: int = 5
    final_context_top_k: int = 4

    embedding_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
    decomposition_model_name: str = "google/flan-t5-small"
    reader_model_name: str = "deepset/roberta-base-squad2"

    max_decomposition_tokens: int = 64
    max_reader_context_characters: int = 2800

    use_hybrid_retrieval: bool = True
    dense_weight: float = 0.6
    bm25_weight: float = 0.4

config = ExperimentConfig()
print(config)

Using device: cuda
ExperimentConfig(train_examples_for_corpus=1200, validation_examples_for_corpus=300, evaluation_examples=120, retrieval_top_k=5, final_context_top_k=4, embedding_model_name='sentence-transformers/all-MiniLM-L6-v2', decomposition_model_name='google/flan-t5-small', reader_model_name='deepset/roberta-base-squad2', max_decomposition_tokens=64, max_reader_context_characters=2800, use_hybrid_retrieval=True, dense_weight=0.6, bm25_weight=0.4)


In [20]:
nlp = spacy.load("en_core_web_sm")

In [21]:
dataset = load_dataset("hotpotqa/hotpot_qa", "distractor")

train_split = dataset["train"]
validation_split = dataset["validation"]

print(train_split)
print(validation_split[0].keys())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

distractor/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/validation-00000-of-00001.par(…):   0%|          | 0.00/27.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
    num_rows: 90447
})
dict_keys(['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'])


In [22]:
sample_example = validation_split[0]
print("Question:", sample_example["question"])
print("Answer:", sample_example["answer"])
print("Type:", sample_example["type"])
print("Supporting facts:", sample_example["supporting_facts"])
print("Context titles:", sample_example["context"]["title"][:3])

Question: Were Scott Derrickson and Ed Wood of the same nationality?
Answer: yes
Type: comparison
Supporting facts: {'title': ['Scott Derrickson', 'Ed Wood'], 'sent_id': [0, 0]}
Context titles: ['Ed Wood (film)', 'Scott Derrickson', 'Woodson, Arkansas']


In [23]:
def join_sentences_into_paragraph(sentences: List[str]) -> str:
    return " ".join(sentence.strip() for sentence in sentences if sentence.strip()).strip()


def build_paragraph_records_from_example(example: Dict[str, Any], source_split_name: str, example_index: int) -> List[Dict[str, Any]]:
    paragraph_records = []

    context_titles = example["context"]["title"]
    context_sentences = example["context"]["sentences"]

    supporting_titles = set(example["supporting_facts"]["title"])

    for title, sentence_list in zip(context_titles, context_sentences):
        paragraph_text = join_sentences_into_paragraph(sentence_list)
        if not paragraph_text:
            continue

        paragraph_records.append(
            {
                "paragraph_id": f"{source_split_name}_{example_index}_{title}",
                "title": title,
                "text": paragraph_text,
                "is_supporting_title_for_this_example": title in supporting_titles,
                "source_example_id": example["id"], # Fixed key from _id to id
            }
        )

    return paragraph_records


def build_shared_corpus(
    train_data,
    validation_data,
    train_limit: int,
    validation_limit: int
) -> List[Dict[str, Any]]:
    deduplicated_paragraphs = {}

    for split_name, split_data, limit in [
        ("train", train_data, train_limit),
        ("validation", validation_data, validation_limit),
    ]:
        for example_index in range(limit):
            example = split_data[example_index]
            paragraph_records = build_paragraph_records_from_example(example, split_name, example_index)

            for paragraph_record in paragraph_records:
                deduplication_key = (
                    paragraph_record["title"].strip().lower(),
                    paragraph_record["text"].strip().lower()
                )
                if deduplication_key not in deduplicated_paragraphs:
                    deduplicated_paragraphs[deduplication_key] = paragraph_record

    return list(deduplicated_paragraphs.values())

In [ ]:
import re
import string
from collections import Counter
from typing import List, Dict, Tuple, Any
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# 1. Restore dataset splits
train_split = dataset['train']
validation_split = dataset['validation']

# 2. Load Evaluation Metric
squad_metric = evaluate.load('squad')

# 3. Load Models and Pipelines
decomposition_tokenizer = AutoTokenizer.from_pretrained(config.decomposition_model_name)
decomposition_model = AutoModelForSeq2SeqLM.from_pretrained(config.decomposition_model_name)

reader_pipeline = pipeline(
    'question-answering',
    model=config.reader_model_name,
    tokenizer=config.reader_model_name,
    device=device
)

print('Environment successfully initialized. You can now run the evaluation cells.')

In [ ]:
import re
import string
from collections import Counter
from typing import List, Dict, Tuple, Any
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# 1. Restore dataset splits
train_split = dataset['train']
validation_split = dataset['validation']

# 2. Load Evaluation Metric
squad_metric = evaluate.load('squad')

# 3. Load Models and Pipelines
decomposition_tokenizer = AutoTokenizer.from_pretrained(config.decomposition_model_name)
decomposition_model = AutoModelForSeq2SeqLM.from_pretrained(config.decomposition_model_name)

reader_pipeline = pipeline(
    'question-answering',
    model=config.reader_model_name,
    tokenizer=config.reader_model_name,
    device=device
)

print('Environment successfully initialized. You can now run the evaluation cells.')

In [ ]:
import re
import string
from collections import Counter
from typing import List, Dict, Tuple, Any
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# 1. Restore dataset splits
train_split = dataset['train']
validation_split = dataset['validation']

# 2. Load Evaluation Metric
squad_metric = evaluate.load('squad')

# 3. Load Models and Pipelines
decomposition_tokenizer = AutoTokenizer.from_pretrained(config.decomposition_model_name)
decomposition_model = AutoModelForSeq2SeqLM.from_pretrained(config.decomposition_model_name)

reader_pipeline = pipeline(
    'question-answering',
    model=config.reader_model_name,
    tokenizer=config.reader_model_name,
    device=device
)

print('Environment successfully initialized. You can now run the evaluation cells.')

In [ ]:
import re
import string
from collections import Counter
from typing import List, Dict, Tuple, Any
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# 1. Restore dataset splits
train_split = dataset['train']
validation_split = dataset['validation']

# 2. Load Evaluation Metric
squad_metric = evaluate.load('squad')

# 3. Load Models and Pipelines
decomposition_tokenizer = AutoTokenizer.from_pretrained(config.decomposition_model_name)
decomposition_model = AutoModelForSeq2SeqLM.from_pretrained(config.decomposition_model_name)

reader_pipeline = pipeline(
    'question-answering',
    model=config.reader_model_name,
    tokenizer=config.reader_model_name,
    device=device
)

print('Environment successfully initialized. You can now run the evaluation cells.')

In [ ]:
import re
import string
from collections import Counter
from typing import List, Dict, Tuple, Any
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# 1. Restore dataset splits
train_split = dataset['train']
validation_split = dataset['validation']

# 2. Load Evaluation Metric
squad_metric = evaluate.load('squad')

# 3. Load Models and Pipelines
decomposition_tokenizer = AutoTokenizer.from_pretrained(config.decomposition_model_name)
decomposition_model = AutoModelForSeq2SeqLM.from_pretrained(config.decomposition_model_name)

reader_pipeline = pipeline(
    'question-answering',
    model=config.reader_model_name,
    tokenizer=config.reader_model_name,
    device=device
)

print('Environment successfully initialized. You can now run the evaluation cells.')

In [ ]:
import re
import string
from collections import Counter
from typing import List, Dict, Tuple, Any
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# 1. Restore dataset splits
train_split = dataset['train']
validation_split = dataset['validation']

# 2. Load Evaluation Metric
squad_metric = evaluate.load('squad')

# 3. Load Models and Pipelines
decomposition_tokenizer = AutoTokenizer.from_pretrained(config.decomposition_model_name)
decomposition_model = AutoModelForSeq2SeqLM.from_pretrained(config.decomposition_model_name)

reader_pipeline = pipeline(
    'question-answering',
    model=config.reader_model_name,
    tokenizer=config.reader_model_name,
    device=device
)

print('Environment successfully initialized. You can now run the evaluation cells.')

In [ ]:
import re
import string
from collections import Counter
from typing import List, Dict, Tuple, Any
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# Restore dataset splits
train_split = dataset['train']
validation_split = dataset['validation']

# Load Evaluation Metric
squad_metric = evaluate.load('squad')

# Initialize Models and Pipelines
decomposition_tokenizer = AutoTokenizer.from_pretrained(config.decomposition_model_name)
decomposition_model = AutoModelForSeq2SeqLM.from_pretrained(config.decomposition_model_name)

reader_pipeline = pipeline(
    'question-answering',
    model=config.reader_model_name,
    tokenizer=config.reader_model_name,
    device=device
)

print('Environment successfully initialized. You can now run the RAG pipelines.')

In [ ]:
import re
import string
from collections import Counter
from typing import List, Dict, Tuple, Any
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# Restore dataset splits
train_split = dataset['train']
validation_split = dataset['validation']

# Load Evaluation Metric
squad_metric = evaluate.load('squad')

# Initialize Models and Pipelines
decomposition_tokenizer = AutoTokenizer.from_pretrained(config.decomposition_model_name)
decomposition_model = AutoModelForSeq2SeqLM.from_pretrained(config.decomposition_model_name)

reader_pipeline = pipeline(
    'question-answering',
    model=config.reader_model_name,
    tokenizer=config.reader_model_name,
    device=device
)

print('Environment successfully initialized. You can now run the RAG pipelines.')

In [ ]:
import re
import string
from collections import Counter
from typing import List, Dict, Tuple, Any
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# 1. Initialize dataset variables
train_split = dataset['train']
validation_split = dataset['validation']

# 2. Load Evaluation Metric
squad_metric = evaluate.load('squad')

# 3. Load Models and Pipelines
decomposition_tokenizer = AutoTokenizer.from_pretrained(config.decomposition_model_name)
decomposition_model = AutoModelForSeq2SeqLM.from_pretrained(config.decomposition_model_name)

reader_pipeline = pipeline(
    'question-answering',
    model=config.reader_model_name,
    tokenizer=config.reader_model_name,
    device=device
)

print('Environment successfully initialized.')

In [24]:
shared_paragraph_corpus = build_shared_corpus(
    train_data=train_split,
    validation_data=validation_split,
    train_limit=config.train_examples_for_corpus,
    validation_limit=config.validation_examples_for_corpus,
)

print("Number of shared corpus paragraphs:", len(shared_paragraph_corpus))
print(shared_paragraph_corpus[0])

Number of shared corpus paragraphs: 14553
{'paragraph_id': 'train_0_Radio City (Indian radio station)', 'title': 'Radio City (Indian radio station)', 'text': "Radio City is India's first private FM radio station and was started on 3 July 2001. It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003). It plays Hindi, English and regional songs. It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007. Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features. The Radio station currently plays a mix of Hindi and Regional music. Abraham Thomas is the CEO of the company.", 'is_supporting_title_for_this_example': False, 'source_example_id': '5a7a06935542990198eaf050'}


In [25]:
paragraph_texts = [record["text"] for record in shared_paragraph_corpus]
paragraph_titles = [record["title"] for record in shared_paragraph_corpus]

def simple_tokenize_for_bm25(text: str) -> List[str]:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()

bm25_corpus_tokens = [simple_tokenize_for_bm25(text) for text in paragraph_texts]
bm25_index = BM25Okapi(bm25_corpus_tokens)

print("BM25 index ready.")

BM25 index ready.


In [26]:
embedding_model = SentenceTransformer(config.embedding_model_name)

paragraph_embeddings = embedding_model.encode(
    paragraph_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Paragraph embeddings shape:", paragraph_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/228 [00:00<?, ?it/s]

Paragraph embeddings shape: (14553, 384)


In [42]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

decomposition_tokenizer = AutoTokenizer.from_pretrained(config.decomposition_model_name)
decomposition_model = AutoModelForSeq2SeqLM.from_pretrained(config.decomposition_model_name)

reader_pipeline = pipeline(
    "question-answering",
    model=config.reader_model_name,
    tokenizer=config.reader_model_name,
    device=device
)

print("Models loaded successfully.")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models loaded successfully.


In [43]:
import evaluate
import re
import string
from collections import Counter
squad_metric = evaluate.load("squad")

def normalize_answer_text(text: str) -> str:
    def remove_articles(value: str) -> str:
        return re.sub(r"\b(a|an|the)\b", " ", value)

    def remove_punctuation(value: str) -> str:
        return "".join(character for character in value if character not in set(string.punctuation))

    def white_space_fix(value: str) -> str:
        return " ".join(value.split())

    def lower(value: str) -> str:
        return value.lower()

    return white_space_fix(remove_articles(remove_punctuation(lower(text))))

def compute_exact_match(prediction: str, gold_answer: str) -> float:
    return float(normalize_answer_text(prediction) == normalize_answer_text(gold_answer))

def compute_token_f1(prediction: str, gold_answer: str) -> float:
    normalized_prediction = normalize_answer_text(prediction).split()
    normalized_gold = normalize_answer_text(gold_answer).split()

    if len(normalized_prediction) == 0 and len(normalized_gold) == 0:
        return 1.0
    if len(normalized_prediction) == 0 or len(normalized_gold) == 0:
        return 0.0

    overlapping_tokens = Counter(normalized_prediction) & Counter(normalized_gold)
    overlap_count = sum(overlapping_tokens.values())

    if overlap_count == 0:
        return 0.0

    precision = overlap_count / len(normalized_prediction)
    recall = overlap_count / len(normalized_gold)

    return 2 * precision * recall / (precision + recall)

In [44]:
from typing import List, Dict, Tuple, Any
import re

pronoun_set = {
    "he", "she", "it", "they", "them", "his", "her", "their",
    "this", "that", "these", "those", "its", "him"
}

def extract_named_entities(text: str) -> List[str]:
    document = nlp(text)
    extracted_entities = []

    for entity in document.ents:
        cleaned_entity = entity.text.strip()
        if cleaned_entity and cleaned_entity.lower() not in pronoun_set:
            extracted_entities.append(cleaned_entity)

    deduplicated_entities = []
    seen_entities = set()

    for entity in extracted_entities:
        entity_key = entity.lower()
        if entity_key not in seen_entities:
            deduplicated_entities.append(entity)
            seen_entities.add(entity_key)

    return deduplicated_entities

def extract_capitalized_phrases_as_fallback(text: str) -> List[str]:
    candidate_phrases = re.findall(r"(?:[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)", text)
    deduplicated_candidates = []

    seen_candidates = set()
    for phrase in candidate_phrases:
        phrase_key = phrase.lower()
        if phrase_key not in seen_candidates:
            deduplicated_candidates.append(phrase)
            seen_candidates.add(phrase_key)

    return deduplicated_candidates

def get_anchor_entities_from_question(question_text: str) -> List[str]:
    named_entities = extract_named_entities(question_text)
    if named_entities:
        return named_entities
    return extract_capitalized_phrases_as_fallback(question_text)

In [30]:
def parse_decomposition_output(raw_generation: str) -> List[str]:
    raw_lines = [line.strip() for line in raw_generation.split("\n") if line.strip()]
    cleaned_questions = []

    for line in raw_lines:
        line = re.sub(r"^\d+[\).\-\:]\s*", "", line).strip()
        if line and "?" not in line:
            line = line + "?"
        if line:
            cleaned_questions.append(line)

    # Keep at most 3 sub-questions
    cleaned_questions = cleaned_questions[:3]

    if len(cleaned_questions) == 0:
        return []

    return cleaned_questions


def decompose_question_with_small_model(question_text: str) -> List[str]:
    prompt = (
        "Break the following multi-hop question into 2 short factual sub-questions. "
        "The sub-questions should help retrieve evidence step by step.\n\n"
        f"Question: {question_text}\n\n"
        "Output format:\n"
        "1. ...\n"
        "2. ..."
    )

    input_ids = decomposition_tokenizer(prompt, return_tensors="pt", truncation=True).input_ids
    generated_ids = decomposition_model.generate(
        input_ids=input_ids,
        max_new_tokens=config.max_decomposition_tokens,
        do_sample=False
    )

    raw_output = decomposition_tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    sub_questions = parse_decomposition_output(raw_output)

    if len(sub_questions) < 2:
        return [question_text]

    return sub_questions

In [31]:
def retrieve_with_bm25(query_text: str, top_k: int) -> List[Tuple[int, float]]:
    tokenized_query = simple_tokenize_for_bm25(query_text)
    bm25_scores = bm25_index.get_scores(tokenized_query)
    ranked_indices = np.argsort(bm25_scores)[::-1][:top_k]
    return [(int(index), float(bm25_scores[index])) for index in ranked_indices]


def retrieve_with_dense_embeddings(query_text: str, top_k: int) -> List[Tuple[int, float]]:
    query_embedding = embedding_model.encode([query_text], normalize_embeddings=True)
    similarity_scores = cosine_similarity(query_embedding, paragraph_embeddings)[0]
    ranked_indices = np.argsort(similarity_scores)[::-1][:top_k]
    return [(int(index), float(similarity_scores[index])) for index in ranked_indices]


def retrieve_hybrid(query_text: str, top_k: int) -> List[Dict[str, Any]]:
    dense_candidates = retrieve_with_dense_embeddings(query_text, top_k=top_k * 3)
    bm25_candidates = retrieve_with_bm25(query_text, top_k=top_k * 3)

    dense_score_map = {index: score for index, score in dense_candidates}
    bm25_score_map = {index: score for index, score in bm25_candidates}

    all_candidate_indices = set(dense_score_map.keys()) | set(bm25_score_map.keys())

    # Normalize BM25 scores locally
    if bm25_score_map:
        bm25_values = np.array(list(bm25_score_map.values()))
        bm25_min, bm25_max = bm25_values.min(), bm25_values.max()
    else:
        bm25_min, bm25_max = 0.0, 1.0

    merged_candidates = []
    for paragraph_index in all_candidate_indices:
        dense_score = dense_score_map.get(paragraph_index, 0.0)
        bm25_score = bm25_score_map.get(paragraph_index, 0.0)

        if bm25_max > bm25_min:
            normalized_bm25_score = (bm25_score - bm25_min) / (bm25_max - bm25_min)
        else:
            normalized_bm25_score = 0.0

        combined_score = (
            config.dense_weight * dense_score
            + config.bm25_weight * normalized_bm25_score
        )

        merged_candidates.append(
            {
                "paragraph_index": paragraph_index,
                "combined_score": combined_score,
                "dense_score": dense_score,
                "bm25_score": bm25_score,
                "title": shared_paragraph_corpus[paragraph_index]["title"],
                "text": shared_paragraph_corpus[paragraph_index]["text"],
            }
        )

    merged_candidates.sort(key=lambda item: item["combined_score"], reverse=True)
    return merged_candidates[:top_k]


def retrieve_paragraphs(query_text: str, top_k: int) -> List[Dict[str, Any]]:
    if config.use_hybrid_retrieval:
        return retrieve_hybrid(query_text, top_k=top_k)

    dense_candidates = retrieve_with_dense_embeddings(query_text, top_k=top_k)
    retrieved_paragraphs = []

    for paragraph_index, score in dense_candidates:
        retrieved_paragraphs.append(
            {
                "paragraph_index": paragraph_index,
                "combined_score": score,
                "dense_score": score,
                "bm25_score": None,
                "title": shared_paragraph_corpus[paragraph_index]["title"],
                "text": shared_paragraph_corpus[paragraph_index]["text"],
            }
        )

    return retrieved_paragraphs

In [32]:
def contains_pronoun_reference(question_text: str) -> bool:
    lowered_question = question_text.lower()
    tokens = re.findall(r"\b\w+\b", lowered_question)
    return any(token in pronoun_set for token in tokens)


def most_recent_entity(entity_memory: List[str]) -> str:
    if len(entity_memory) == 0:
        return ""
    return entity_memory[-1]


def replace_pronouns_with_entity(question_text: str, replacement_entity: str) -> str:
    if not replacement_entity:
        return question_text

    rewritten_question = question_text

    pronoun_patterns = [
        r"\bhe\b", r"\bshe\b", r"\bit\b", r"\bthey\b", r"\bthem\b",
        r"\bhis\b", r"\bher\b", r"\btheir\b", r"\bthis\b", r"\bthat\b"
    ]

    for pattern in pronoun_patterns:
        rewritten_question = re.sub(
            pattern,
            replacement_entity,
            rewritten_question,
            flags=re.IGNORECASE
        )

    return rewritten_question


def append_missing_anchor_entity(question_text: str, anchor_entities: List[str]) -> str:
    question_lower = question_text.lower()

    for anchor_entity in anchor_entities:
        if anchor_entity.lower() not in question_lower:
            return f"{question_text} about {anchor_entity}"

    return question_text


def rewrite_sub_question_with_entity_memory(
    original_question_text: str,
    current_sub_question: str,
    entity_memory: List[str]
) -> str:
    rewritten_question = current_sub_question
    anchor_entities = get_anchor_entities_from_question(original_question_text)

    if contains_pronoun_reference(rewritten_question):
        rewritten_question = replace_pronouns_with_entity(
            rewritten_question,
            most_recent_entity(entity_memory)
        )

    rewritten_question = append_missing_anchor_entity(rewritten_question, anchor_entities)

    return rewritten_question

In [33]:
def build_reader_context(retrieved_paragraphs: List[Dict[str, Any]], max_characters: int) -> str:
    combined_sections = []

    for paragraph in retrieved_paragraphs:
        section_text = f"Title: {paragraph['title']}\n{paragraph['text']}"
        combined_sections.append(section_text)

    combined_context = "\n\n".join(combined_sections)
    return combined_context[:max_characters]


def answer_question_from_retrieved_context(question_text: str, retrieved_paragraphs: List[Dict[str, Any]]) -> Dict[str, Any]:
    reader_context = build_reader_context(
        retrieved_paragraphs,
        max_characters=config.max_reader_context_characters
    )

    if not reader_context.strip():
        return {
            "answer": "",
            "score": 0.0,
            "context_used": reader_context
        }

    reader_output = reader_pipeline(
        question=question_text,
        context=reader_context
    )

    return {
        "answer": reader_output["answer"],
        "score": float(reader_output["score"]),
        "context_used": reader_context
    }

In [34]:
def run_vanilla_rag(question_text: str) -> Dict[str, Any]:
    retrieved_paragraphs = retrieve_paragraphs(
        query_text=question_text,
        top_k=config.final_context_top_k
    )

    reader_result = answer_question_from_retrieved_context(
        question_text=question_text,
        retrieved_paragraphs=retrieved_paragraphs
    )

    return {
        "method_name": "vanilla_rag",
        "sub_questions": [question_text],
        "rewritten_sub_questions": [question_text],
        "retrieved_paragraphs": retrieved_paragraphs,
        "predicted_answer": reader_result["answer"],
        "reader_confidence": reader_result["score"],
    }


def run_decomposition_rag(question_text: str) -> Dict[str, Any]:
    sub_questions = decompose_question_with_small_model(question_text)
    all_retrieved_paragraphs = []

    for sub_question in sub_questions:
        retrieved_for_sub_question = retrieve_paragraphs(
            query_text=sub_question,
            top_k=config.retrieval_top_k
        )
        all_retrieved_paragraphs.extend(retrieved_for_sub_question)

    # Deduplicate by paragraph index and keep highest score
    best_paragraphs_by_index = {}
    for paragraph in all_retrieved_paragraphs:
        paragraph_index = paragraph["paragraph_index"]
        if paragraph_index not in best_paragraphs_by_index:
            best_paragraphs_by_index[paragraph_index] = paragraph
        else:
            if paragraph["combined_score"] > best_paragraphs_by_index[paragraph_index]["combined_score"]:
                best_paragraphs_by_index[paragraph_index] = paragraph

    merged_retrieved_paragraphs = list(best_paragraphs_by_index.values())
    merged_retrieved_paragraphs.sort(key=lambda item: item["combined_score"], reverse=True)
    merged_retrieved_paragraphs = merged_retrieved_paragraphs[:config.final_context_top_k]

    reader_result = answer_question_from_retrieved_context(
        question_text=question_text,
        retrieved_paragraphs=merged_retrieved_paragraphs
    )

    return {
        "method_name": "decomposition_rag",
        "sub_questions": sub_questions,
        "rewritten_sub_questions": sub_questions,
        "retrieved_paragraphs": merged_retrieved_paragraphs,
        "predicted_answer": reader_result["answer"],
        "reader_confidence": reader_result["score"],
    }


def run_entity_aware_progressive_rag(question_text: str) -> Dict[str, Any]:
    sub_questions = decompose_question_with_small_model(question_text)
    entity_memory = get_anchor_entities_from_question(question_text).copy()

    rewritten_sub_questions = []
    collected_retrieved_paragraphs = []
    intermediate_answers = []

    for sub_question in sub_questions:
        rewritten_sub_question = rewrite_sub_question_with_entity_memory(
            original_question_text=question_text,
            current_sub_question=sub_question,
            entity_memory=entity_memory
        )
        rewritten_sub_questions.append(rewritten_sub_question)

        retrieved_for_sub_question = retrieve_paragraphs(
            query_text=rewritten_sub_question,
            top_k=config.retrieval_top_k
        )
        collected_retrieved_paragraphs.extend(retrieved_for_sub_question)

        sub_question_reader_result = answer_question_from_retrieved_context(
            question_text=rewritten_sub_question,
            retrieved_paragraphs=retrieved_for_sub_question[:3]
        )
        intermediate_answer = sub_question_reader_result["answer"]
        intermediate_answers.append(intermediate_answer)

        discovered_entities = extract_named_entities(intermediate_answer)
        for entity in discovered_entities:
            if entity.lower() not in {known_entity.lower() for known_entity in entity_memory}:
                entity_memory.append(entity)

    best_paragraphs_by_index = {}
    for paragraph in collected_retrieved_paragraphs:
        paragraph_index = paragraph["paragraph_index"]
        if paragraph_index not in best_paragraphs_by_index:
            best_paragraphs_by_index[paragraph_index] = paragraph
        else:
            if paragraph["combined_score"] > best_paragraphs_by_index[paragraph_index]["combined_score"]:
                best_paragraphs_by_index[paragraph_index] = paragraph

    merged_retrieved_paragraphs = list(best_paragraphs_by_index.values())
    merged_retrieved_paragraphs.sort(key=lambda item: item["combined_score"], reverse=True)
    merged_retrieved_paragraphs = merged_retrieved_paragraphs[:config.final_context_top_k]

    final_reader_result = answer_question_from_retrieved_context(
        question_text=question_text,
        retrieved_paragraphs=merged_retrieved_paragraphs
    )

    return {
        "method_name": "entity_aware_progressive_rag",
        "sub_questions": sub_questions,
        "rewritten_sub_questions": rewritten_sub_questions,
        "intermediate_answers": intermediate_answers,
        "retrieved_paragraphs": merged_retrieved_paragraphs,
        "predicted_answer": final_reader_result["answer"],
        "reader_confidence": final_reader_result["score"],
        "entity_memory": entity_memory,
    }

In [35]:
def compute_supporting_title_recall(example: Dict[str, Any], retrieved_paragraphs: List[Dict[str, Any]]) -> float:
    gold_titles = set(example["supporting_facts"]["title"])
    retrieved_titles = set(paragraph["title"] for paragraph in retrieved_paragraphs)

    if len(gold_titles) == 0:
        return 0.0

    matched_title_count = len(gold_titles & retrieved_titles)
    return matched_title_count / len(gold_titles)

In [36]:
def evaluate_method_on_hotpot_subset(method_function, evaluation_limit: int) -> pd.DataFrame:
    result_rows = []

    for example_index in range(evaluation_limit):
        example = validation_split[example_index]

        question_text = example["question"]
        gold_answer = example["answer"]

        try:
            method_result = method_function(question_text)
        except Exception as execution_error:
            print(f"Skipping example {example_index} due to error: {execution_error}")
            continue

        predicted_answer = method_result["predicted_answer"]

        exact_match_value = compute_exact_match(predicted_answer, gold_answer)
        token_f1_value = compute_token_f1(predicted_answer, gold_answer)
        title_recall_value = compute_supporting_title_recall(
            example=example,
            retrieved_paragraphs=method_result["retrieved_paragraphs"]
        )

        result_rows.append(
            {
                "example_index": example_index,
                "question": question_text,
                "gold_answer": gold_answer,
                "predicted_answer": predicted_answer,
                "method_name": method_result["method_name"],
                "exact_match": exact_match_value,
                "token_f1": token_f1_value,
                "supporting_title_recall": title_recall_value,
                "reader_confidence": method_result.get("reader_confidence", 0.0),
                "question_type": example["type"],
                "sub_questions": method_result.get("sub_questions", []),
                "rewritten_sub_questions": method_result.get("rewritten_sub_questions", []),
                "retrieved_titles": [paragraph["title"] for paragraph in method_result["retrieved_paragraphs"]],
            }
        )

        if (example_index + 1) % 10 == 0:
            print(f"Processed {example_index + 1}/{evaluation_limit} examples for {method_result['method_name']}")

    return pd.DataFrame(result_rows)

In [37]:
vanilla_results_df = evaluate_method_on_hotpot_subset(
    method_function=run_vanilla_rag,
    evaluation_limit=config.evaluation_examples
)

decomposition_results_df = evaluate_method_on_hotpot_subset(
    method_function=run_decomposition_rag,
    evaluation_limit=config.evaluation_examples
)

entity_aware_results_df = evaluate_method_on_hotpot_subset(
    method_function=run_entity_aware_progressive_rag,
    evaluation_limit=config.evaluation_examples
)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Processed 10/120 examples for vanilla_rag
Processed 20/120 examples for vanilla_rag
Processed 30/120 examples for vanilla_rag
Processed 40/120 examples for vanilla_rag
Processed 50/120 examples for vanilla_rag
Processed 60/120 examples for vanilla_rag
Processed 70/120 examples for vanilla_rag
Processed 80/120 examples for vanilla_rag
Processed 90/120 examples for vanilla_rag
Processed 100/120 examples for vanilla_rag
Processed 110/120 examples for vanilla_rag
Processed 120/120 examples for vanilla_rag
Processed 10/120 examples for decomposition_rag
Processed 20/120 examples for decomposition_rag
Processed 30/120 examples for decomposition_rag
Processed 40/120 examples for decomposition_rag
Processed 50/120 examples for decomposition_rag
Processed 60/120 examples for decomposition_rag
Processed 70/120 examples for decomposition_rag
Processed 80/120 examples for decomposition_rag
Processed 90/120 examples for decomposition_rag
Processed 100/120 examples for decomposition_rag
Processed 11

In [38]:
def summarize_result_frame(result_frame: pd.DataFrame) -> Dict[str, float]:
    return {
        "EM": result_frame["exact_match"].mean(),
        "F1": result_frame["token_f1"].mean(),
        "Supporting_Title_Recall": result_frame["supporting_title_recall"].mean(),
        "Average_Reader_Confidence": result_frame["reader_confidence"].mean(),
        "Number_of_Examples": len(result_frame),
    }

summary_table = pd.DataFrame(
    [
        {"method": "Vanilla RAG", **summarize_result_frame(vanilla_results_df)},
        {"method": "Decomposition RAG", **summarize_result_frame(decomposition_results_df)},
        {"method": "Entity-Aware Progressive RAG", **summarize_result_frame(entity_aware_results_df)},
    ]
)

summary_table

,method,EM,F1,Supporting_Title_Recall,Average_Reader_Confidence,Number_of_Examples
0,Vanilla RAG,0.241667,0.295584,0.654167,0.199986,120
1,Decomposition RAG,0.241667,0.291418,0.658333,0.214796,120
2,Entity-Aware Progressive RAG,0.241667,0.296973,0.658333,0.219329,120


In [39]:
comparison_frame = vanilla_results_df.merge(
    decomposition_results_df,
    on="example_index",
    suffixes=("_vanilla", "_decomposition")
).merge(
    entity_aware_results_df,
    on="example_index"
)

comparison_frame = comparison_frame.rename(
    columns={
        "exact_match": "exact_match_entity_aware",
        "token_f1": "token_f1_entity_aware",
        "supporting_title_recall": "supporting_title_recall_entity_aware",
        "predicted_answer": "predicted_answer_entity_aware",
        "question": "question_entity_aware",
        "gold_answer": "gold_answer_entity_aware",
        "rewritten_sub_questions": "rewritten_sub_questions_entity_aware",
    }
)

comparison_frame["entity_aware_f1_gain_over_decomposition"] = (
    comparison_frame["token_f1_entity_aware"] - comparison_frame["token_f1_decomposition"]
)

comparison_frame["entity_aware_f1_gain_over_vanilla"] = (
    comparison_frame["token_f1_entity_aware"] - comparison_frame["token_f1_vanilla"]
)

helpful_examples = comparison_frame.sort_values(
    by="entity_aware_f1_gain_over_decomposition",
    ascending=False
).head(10)

helpful_examples[
    [
        "example_index",
        "question_entity_aware",
        "gold_answer_entity_aware",
        "predicted_answer_vanilla",
        "predicted_answer_decomposition",
        "predicted_answer_entity_aware",
        "token_f1_vanilla",
        "token_f1_decomposition",
        "token_f1_entity_aware",
        "rewritten_sub_questions_entity_aware",
    ]
]

,example_index,question_entity_aware,gold_answer_entity_aware,predicted_answer_vanilla,predicted_answer_decomposition,predicted_answer_entity_aware,token_f1_vanilla,token_f1_decomposition,token_f1_entity_aware,rewritten_sub_questions_entity_aware
5,5,2014 S/S is the debut album of a South Korean ...,YG Entertainment,FNC Entertainment,J. Tune Camp,YG Entertainment,0.5,0.0,1.000000,[2014 S/S is the debut album of a South Korean...
67,67,What is the name of the singer who's song was ...,Usher,Taylor Swift,Taylor Swift,Usher,0.0,0.0,1.000000,[What is the name of the singer who's song was...
50,50,Which year and which conference was the 14th s...,2009 Big 12 Conference,Pac-12,Pac-12,Big 12,0.0,0.0,0.666667,[Which year and which conference was the 14th ...
2,2,"What science fantasy young adult series, told ...",Animorphs,Animorphs,Animorphs,Animorphs,1.0,1.0,1.000000,"[What science fantasy young adult series, told..."
3,3,Are the Laleli Mosque and Esma Sultan Mansion ...,no,Ortaköy,Ortaköy,Ortaköy,0.0,0.0,0.000000,[Are the Laleli Mosque and Esma Sultan Mansion...
4,4,"The director of the romantic comedy ""Big Stone...","Greenwich Village, New York City",Adriana Trigiani,Adriana Trigiani,Adriana Trigiani,0.0,0.0,0.000000,"[The director of the romantic comedy ""Big Ston..."
6,6,Who was known by his stage name Aladin and hel...,Eenasul Fateh,Eenasul Fateh,Eenasul Fateh,Eenasul Fateh,1.0,1.0,1.000000,[Who was known by Aladin stage name Aladin and...
0,0,Were Scott Derrickson and Ed Wood of the same ...,yes,American,American,American,0.0,0.0,0.000000,[Were Scott Derrickson and Ed Wood of the same...
8,8,"Who is older, Annie Morton or Terry Richardson?",Terry Richardson,Lady Gaga,Lady Gaga,Lady Gaga,0.0,0.0,0.000000,"[Who is older, Annie Morton or Terry Richardson?]"
9,9,Are Local H and For Against both from the Unit...,yes,the builder of the house has not been determined,the builder of the house has not been determined,the builder of the house has not been determined,0.0,0.0,0.000000,[Are Local H and For Against both from the Uni...


In [40]:
def grouped_metric_table(result_frame: pd.DataFrame) -> pd.DataFrame:
    return result_frame.groupby("question_type")[["exact_match", "token_f1", "supporting_title_recall"]].mean().reset_index()

print("Vanilla RAG")
display(grouped_metric_table(vanilla_results_df))

print("Decomposition RAG")
display(grouped_metric_table(decomposition_results_df))

print("Entity-Aware Progressive RAG")
display(grouped_metric_table(entity_aware_results_df))

Vanilla RAG


,question_type,exact_match,token_f1,supporting_title_recall
0,bridge,0.281250,0.342695,0.645833
1,comparison,0.083333,0.107143,0.687500


Decomposition RAG


,question_type,exact_match,token_f1,supporting_title_recall
0,bridge,0.281250,0.337486,0.651042
1,comparison,0.083333,0.107143,0.687500


Entity-Aware Progressive RAG


,question_type,exact_match,token_f1,supporting_title_recall
0,bridge,0.281250,0.344431,0.651042
1,comparison,0.083333,0.107143,0.687500


In [41]:
summary_table.to_csv("summary_table.csv", index=False)
vanilla_results_df.to_csv("vanilla_results.csv", index=False)
decomposition_results_df.to_csv("decomposition_results.csv", index=False)
entity_aware_results_df.to_csv("entity_aware_results.csv", index=False)

print("Saved result files.")

Saved result files.
